# Fish Farming / Model Training / Image Conversion

# 🛠 Manual image selection containing cages before training

---

For this step, there were manually selected **all** the images that contain a cage or part of it.

This will depend on the kind of images the model will be trained.


# Libraries Installation

---



In [ ]:
!pip install rasterio

# Variables configuration

---



In [1]:
import os

#Folder containing the (manually) preselected images for training
image_dir = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/raw/sentinel2_rgb_cages'

#Folder to store the selected images based on uniqueness
output_dir = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/raw/sentinel2_rgb_selected'

#PNG converted images from selected based on uniqueness
output_folder_png = "/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed"

#Number of images to be selected based on uniqueness
number_selected_images = 50

#Folder to store 'npy' converted images
output_folder_npy = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy'

#Folder creation
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_folder_png, exist_ok=True)
os.makedirs(output_folder_npy, exist_ok=True)

# Pick images based on uniqueness

---

From selected images and given that some images may look repeated, a uniqueness filter is applied to get the most unique ones.

In [2]:
import os
import shutil
import random
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_distances
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model

# Load pre-trained model to analize uniqueness
base_model = VGG16(weights='imagenet', include_top=False, pooling='avg')
model = Model(inputs=base_model.input, outputs=base_model.output)

def load_and_preprocess_image(path, target_size=(224, 224)):
    img = Image.open(path).convert('RGB').resize(target_size)
    img_array = img_to_array(img)
    img_array = preprocess_input(np.expand_dims(img_array, axis=0))
    return img_array

# Collect image paths
image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.lower().endswith('.tiff')]

# Extract features
features = []
valid_paths = []
for path in image_paths:
    try:
        img_array = load_and_preprocess_image(path)
        feature = model.predict(img_array, verbose=0)[0]
        features.append(feature)
        valid_paths.append(path)
    except Exception as e:
        print(f"Error processing {path}: {e}")

features = np.array(features)

# Compute uniqueness scores
dist_matrix = cosine_distances(features)
uniqueness_scores = dist_matrix.mean(axis=1)

# Select top number_selected_images most unique images
unique_indices = np.argsort(uniqueness_scores)[-(number_selected_images):]
selected_images = [valid_paths[i] for i in unique_indices]

# Move selected images to output folder
for img_path in selected_images:
    filename = os.path.basename(img_path)
    shutil.move(img_path, os.path.join(output_dir, filename))
    print(f"Moved: {filename}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Moved: sentinel2_949.tiff
Moved: sentinel2_851.tiff
Moved: sentinel2_561.tiff
Moved: sentinel2_945.tiff
Moved: sentinel2_1021.tiff
Moved: sentinel2_27.tiff
Moved: sentinel2_279.tiff
Moved: sentinel2_1135.tiff
Moved: sentinel2_759.tiff
Moved: sentinel2_505.tiff
Moved: sentinel2_415.tiff
Moved: sentinel2_553.tiff
Moved: sentinel2_638.tiff
Moved: sentinel2_458.tiff
Moved: sentinel2_895.tiff
Moved: sentinel2_502.tiff
Moved: sentinel2_658.tiff
Moved: sentinel2_545.tiff
Moved: sentinel2_622.tiff
Moved: sentinel2_529.tiff
Moved: sentinel2_690.tiff
Moved: sentinel2_17.tiff
Moved: sentinel2_675.tiff
Moved: sentinel2_647.tiff
Moved: sentinel2_669.tiff
Moved: sentinel2_1070.tiff
Moved: sentinel2_897.tiff
Moved: sentinel2_469.tiff
Moved: sentinel2_294.tiff
Moved: sentinel2_211.tiff
Moved: sentinel2_357.tiff
Moved: sentinel2_173.tiff
Moved: sentinel2_43.tiff
Moved: sentinel2_570.tiff
Moved: sentinel2_911.tiff
Moved: sentinel2_559.tiff
Moved: sentin

# Transforming Images

---



# Converting images to PNG for later Mask process

In [5]:
import cv2
import os
import glob
import numpy as np
import rasterio

input_folder = output_dir

#Function to check if an image is too dark
def is_black_image(image_path, threshold=10):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    avg_brightness = np.mean(image)  # Calculate average pixel brightness
    return avg_brightness < threshold  # Returns True if mostly dark/black

#Function to check if images are completely cover by clouds
def detect_clouds(image_path):
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)  # Detect bright areas

    cloud_percentage = (np.sum(thresh == 255) / thresh.size) * 100  # Calculate coverage
    return cloud_percentage

# Converting TIFF images to PNG
for image_path in glob.glob(os.path.join(input_folder, "*.tiff")):
    '''if is_black_image(image_path) or detect_clouds(image_path) > 50:
        print(f"Skipping black or cloudy image: {image_path}")
        continue
    else:'''
    with rasterio.open(image_path) as src:
        image_data = src.read([1, 2, 3])  # Read RGB bands
        image_data = np.moveaxis(image_data, 0, -1)  # Rearrange dimensions

        # Normalize and convert to uint8
        image_data = (image_data / np.max(image_data) * 255).astype(np.uint8)

        output_file = os.path.join(output_folder_png, os.path.basename(image_path).replace(".tiff", ".png"))
        cv2.imwrite(output_file, image_data)

        print(f"Converted: {output_file}")

Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_17.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_27.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_28.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_43.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_173.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_182.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_rgb_processed/sentinel2_211.png
Converted: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/

# Convert tiff images to npy for training

---



In [6]:
import os
import numpy as np
import tifffile

# Set your folder path
input_folder = output_dir

# Loop over all .tif or .tiff files
for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.tif', '.tiff')):
        tiff_path = os.path.join(input_folder, filename)
        npy_path = os.path.join(output_folder_npy, filename.rsplit('.', 1)[0] + '.npy')

        # Read TIFF image
        image = tifffile.imread(tiff_path)

        # Save as .npy
        np.save(npy_path, image)
        print(f"Saved: {npy_path}")


Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_17.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_27.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_28.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_43.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_173.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_182.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_211.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy/sentinel2_279.npy
Saved: /content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/mode